### **Frankenstein font**
We are going to combine some gliphs from the Jena font and the Brokenscript font.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from fontTools.ttLib import TTFont
from fontTools.pens.cu2quPen import Cu2QuPen
from fontTools.pens.ttGlyphPen import TTGlyphPen

load_dotenv()
project_root = Path(os.environ["PROJECT_ROOT"])

# Source = Missaali (CFF / .otf — cubic Bezier outlines)
# Target = Merge (glyf / .ttf — quadratic Bezier outlines)
font_merge    = TTFont(project_root / "fonts/merged_font_code.ttf")
font_missaali = TTFont(project_root / "fonts/Missaali-Regular.otf")

# Glyph names differ between fonts — look up by Unicode codepoint instead.
src_cmap = font_missaali.getBestCmap()       # {codepoint: glyph_name}
dst_cmap = font_merge.getBestCmap()

# UPM-aware scaling — both fonts use UPM=1000 here so scale==1, but the
# general form lets this work for any font pair.
src_upm = font_missaali["head"].unitsPerEm
dst_upm = font_merge["head"].unitsPerEm
scale = dst_upm / src_upm

src_glyph_set = font_missaali.getGlyphSet()
chars_to_copy = ["ꝛ"]

for ch in chars_to_copy:
    cp = ord(ch)
    if cp not in src_cmap:
        print(f"'{ch}' (U+{cp:04X}) missing from Missaali, skipping")
        continue

    src_name = src_cmap[cp]

    # If Merge already has a glyph for this codepoint, overwrite it.
    # Otherwise create a new glyph slot and wire it into the cmap so the
    # codepoint actually resolves to it.
    if cp in dst_cmap:
        dst_name = dst_cmap[cp]
    else:
        dst_name = f"uni{cp:04X}"
        if dst_name not in font_merge.getGlyphOrder():
            font_merge.getGlyphOrder().append(dst_name)
        for sub in font_merge["cmap"].tables:
            if sub.isUnicode():
                sub.cmap[cp] = dst_name

    # CFF (cubic) -> glyf (quadratic) via cu2qu pen.
    # reverse_direction=True flips winding order: PostScript/CFF use the
    # opposite contour direction from TrueType, otherwise the glyph
    # renders filled inside-out.
    pen = TTGlyphPen(None)
    cu2qu_pen = Cu2QuPen(pen, max_err=1.0, reverse_direction=True)
    src_glyph_set[src_name].draw(cu2qu_pen)

    font_merge["glyf"][dst_name] = pen.glyph()

    # Copy advance width + left sidebearing (scaled if UPMs differ).
    adv, lsb = font_missaali["hmtx"][src_name]
    font_merge["hmtx"][dst_name] = (round(adv * scale), round(lsb * scale))

    print(f"✓ '{ch}': Missaali '{src_name}' → Merge '{dst_name}'")

out_path = project_root / "fonts/merged_font_code_cmpl.ttf"
font_merge.save(out_path)
print(f"\nMerged font saved to: {out_path}")